# **SQL Data Cleaning, Transformation and Analysis: Raw Data of Open-Pit Copper Mine**

As there are no publicly available raw datasets for open-pit copper mine operations, synthetic data representing an open-pit copper mine was generated using generative artificial intelligence.

Generative artificial intelligence was used solely for the creation of the synthetic dataset. All SQL data extraction, cleaning, transformation, and analysis presented in this project are entirely my own work. 

The relational data model for the dataset is shown in the image below and consists of five tables, representing the following information:

1. **`01_energy_sources_raw`** – Energy sources used across the mine's operations <br><br>
   
2. **`02_operation_hierarchy_raw`** – The operations and sub-operations within the mine <br><br>
  
3. **`03_daily_production_raw`** – Daily mine production data <br><br>
   
4. **`04_daily_energy_consumption_raw`** – Daily energy consumption across the mine's operations <br><br>
  
5. **`05_monthly_energy_costs_raw`** – Monthly energy costs associated with the various energy sources used across different mine operations <br><br>
   
![Relational Database](Relational%20Database.png)

The raw datasets contained inconsistent text formatting, mixed date formats, non-numeric values, missing values, different energy units, and values outside expected ranges. SQL was used to clean and transform the datasets before combining them for the final energy, cost and emissions analysis.

## **Table 1 (01_energy_sources_raw) - Data Cleaning & Transformation:**

A new table, **`energy_sources_clean`**, was created from the **`01_energy_sources_raw`** table using the `CREATE TABLE` statement. 

During this process, several data cleaning and transformation steps were applied:

* The `TRIM()` function was used to remove leading and trailing spaces, while the `UPPER()` function was applied to standardise the formatting of energy source names <br><br>

* Energy units were standardised using a `CASE` statement to ensure consistency across the dataset. For example, values such as 'mwh' were converted to 'MWh', while 'litres' was standardised to 'L' <br><br>

* The **`emissions_factor_tco2e_per_unit`** column represents the emissions factor for each energy source, expressed as tonnes of CO₂e per unit of energy consumed. <br><br>As the original emissions factors were reported using different units depending on the energy source, i.e. MWh, GJ, and L, a new column named **`emissions_factor_tco2e_per_MJ`** was created in order to standardise the emissions factor values <br><br>

* A new column, **`MJ_per_unit`**, was also created to store the conversion factor between each energy source's original unit and megajoules (MJ).<br><br>  The conversion factors used were: <br><br> 1 MWh = 3,6000 MJ <br><br> 1L = 40 MJ (Diesel) <br><br> 1 GJ = 1000 MJ <br><br> This conversion factor was then used to calculate the standardised values in the **`emissions_factor_tco2e_per_MJ`** column. <br><br> 

In [3]:
-- TABLE 1 - ENERGY SOURCES
CREATE TABLE energy_sources_clean AS 
SELECT * 
FROM '01_energy_sources_raw.csv';


UPDATE energy_sources_clean
	SET energy_source = UPPER(TRIM(energy_source)),
	
	standard_unit = CASE WHEN standard_unit ILIKE '%mwh%' THEN 'MWh'
	                     WHEN standard_unit ILIKE '%litres%' THEN 'L'
	                	 ELSE standard_unit END;

ALTER TABLE energy_sources_clean
	ADD COLUMN MJ_per_unit NUMERIC;

UPDATE energy_sources_clean
   SET MJ_per_unit = CASE WHEN standard_unit = 'MWh' THEN 3600
	                      WHEN standard_unit = 'L'   THEN 40
	                      WHEN standard_unit = 'GJ'  THEN 1000
	                      ELSE MJ_per_unit END;

ALTER TABLE energy_sources_clean
	ADD COLUMN emissions_factor_tco2e_per_MJ DOUBLE;

UPDATE energy_sources_clean
   SET emissions_factor_tco2e_per_MJ = emissions_factor_tco2e_per_unit/MJ_per_unit;

SELECT *
FROM energy_sources_clean;

,energy_source_id,energy_source,standard_unit,emissions_factor_tco2e_per_unit,MJ_per_unit,emissions_factor_tco2e_per_MJ
0,16706266,GRID ELECTRICITY,MWh,0.69000,3600.0,0.000192
1,28450097,DIESEL,L,0.00268,40.0,0.000067
2,53385195,NATURAL GAS,GJ,0.05150,1000.0,0.000051
3,22757633,SOLAR PPA,MWh,0.03000,3600.0,0.000008


<br>## **Table 2 (02_operation_hierarchy_raw) - Data Cleaning & Transformation:**

A new table, **`operation_hierarchy_clean`**, was created from the **`02_operation_hierarchy_raw`** table using the `CREATE TABLE` statement.

The data cleaning and transformation steps applied were as follows:

* The `TRIM()` function was used to remove leading and trailing spaces for both the **`main_operation`** and **`sub_operation`** columns, while the `UPPER()` function was applied to standardise the formatting of main operation names <br><br>


In [6]:
-- TABLE 2 - OPERATION HIERARCHY
CREATE TABLE operation_hierarchy_clean AS 
SELECT * 
FROM '02_operation_hierarchy_raw.csv';

UPDATE operation_hierarchy_clean 
	SET main_operation = TRIM(UPPER(main_operation)),
	    sub_operation = TRIM(sub_operation);

SELECT *
FROM operation_hierarchy_clean;

,operation_id,main_operation,sub_operation,process_stage_order,operational_area,energy_criticality,typical_schedule
0,91340540,MINE DEVELOPMENT,Grade-control drilling,1,Open pit - ore zones,Medium,24/7
1,24914615,MINE DEVELOPMENT,Blast-hole drilling,2,Open pit - active benches,High,24/7
2,62814762,MINE DEVELOPMENT,Blasting support,3,Open pit - active benches,Medium,Day shift
3,77640743,MINE DEVELOPMENT,Pit dewatering,4,Open pit sumps,High,24/7
4,83276760,MINE DEVELOPMENT,Haul-road maintenance,5,Pit and waste routes,Medium,24/7
5,61354486,MATERIAL MOVEMENT,Excavating and loading,6,Open pit loading faces,High,24/7
6,49569161,MATERIAL MOVEMENT,Ore hauling,7,Pit to ROM pad,High,24/7
7,98042943,MATERIAL MOVEMENT,Waste hauling,8,Pit to waste dumps,High,24/7
8,34819938,MATERIAL MOVEMENT,Dozing and grading,9,Pit and stockpiles,Medium,24/7
9,98788663,MATERIAL MOVEMENT,Stockpile Reclaim,10,ROM stockpile,High,24/7


<br>## **Table 3 (02_operation_hierarchy_raw) - Data Cleaning & Transformation:**

In [8]:
SELECT *
FROM daily_production_clean;

,production_record_id,production_date,operation_id,metric_name,metric_value,unit,shift_reporting
0,11535043,2025-01-01 00:00:00+00:00,98042943,TOTAL MATERIAL MINED,107328.412,t,Daily total
1,77562948,2025-01-01 00:00:00+00:00,49569161,ORE MINED,26985.474,t,Daily total
2,96003108,2025-01-01 00:00:00+00:00,58956654,ORE PROCESSED,29480.207,t,Daily total
3,90389400,2025-01-01 00:00:00+00:00,15289718,COPPER CONCENTRATE PRODUCED,733.293,t,Daily total
4,41973496,2025-01-01 00:00:00+00:00,45614007,HEAD GRADE,0.723,%,Daily total
...,...,...,...,...,...,...,...
2311,28454822,2025-07-02 00:00:00+00:00,45614007,HEAD GRADE,0.658,%,Daily total
2312,32318115,2025-12-21 00:00:00+00:00,49569161,ORE MINED,27652.709,t,Daily total
2313,67615010,2025-05-17 00:00:00+00:00,45769449,COPPER RECOVERY,88.477,%,Daily total
2314,74639440,2025-06-10 00:00:00+00:00,45614007,HEAD GRADE,0.687,%,Daily total
